In [ ]:
CREATE TABLE CERVEZAS (
    CodC TEXT,
    Envase TEXT,
    Capacidad REAL,
    Stock INTEGER
);


In [ ]:
CREATE TABLE BARES (
    CodB TEXT,
    Cif TEXT,
    Nombre TEXT,
    Localidad TEXT
);

In [ ]:
CREATE TABLE EMPLEADOS (
    CodE INTEGER,
    Nombre TEXT,
    Sueldo INTEGER
);

In [ ]:
CREATE TABLE REPARTO (
    CodE INTEGER,
    CodB TEXT,
    CodC TEXT,
    Fecha TEXT,
    Cantidad INTEGER
);

In [ ]:
SELECT name
FROM sqlite_master
WHERE type = 'table';# проверка, что таблицы созданы

In [ ]:
INSERT INTO CERVEZAS (CodC, Envase, Capacidad, Stock)
VALUES
('01', 'Botella', 0.2, 3600),
('02', 'Botella', 0.33, 1200),
('03', 'Lata', 0.33, 2400),
('04', 'Botella', 1, 288),
('05', 'Barril', 60, 30);

SELECT *
FROM CERVEZAS;

DELETE FROM CERVEZAS;

In [ ]:
INSERT INTO BARES (CodB, Cif, Nombre, Localidad)
VALUES
('001', '11111111X', 'Stop', 'Villa Botijo'),
('002', '22222222Y', 'Las Vegas', 'Villa Botijo'),
('003', NULL, 'Club Social', 'Las Ranas'),
('004', '33333333Z', 'Otra Ronda', 'La Esponja');

SELECT *
FROM BARES;

In [ ]:
INSERT INTO EMPLEADOS (CodE, Nombre, Sueldo)
VALUES
(1, 'Prudencio Caminero', 120000),
(2, 'Vicente Merario', 110000),
(3, 'Valentin Siempre', 100000);

SELECT *
FROM EMPLEADOS;

In [ ]:
INSERT INTO REPARTO (CodE, CodB, CodC, Fecha, Cantidad)
VALUES
(1, '001', '01', '10/21/05', 240),
(1, '001', '02', '10/21/05', 48),
(1, '002', '03', '10/22/05', 60),
(1, '004', '05', '10/22/05', 4),
(2, '002', '03', '10/22/05', 48),
(2, '002', '05', '10/23/05', 2),
(2, '004', '01', '10/23/05', 480),
(2, '004', '02', '10/24/05', 72),
(3, '003', '03', '10/24/05', 48),
(3, '003', '04', '10/25/05', 20);

SELECT *
FROM REPARTO;

In [ ]:
#1. Obtener el nombre de los empleados que hayan repartido al bar **Stop** durante la semana del 17 al 23 de octubre de 2005.

SELECT DISTINCT 
	e.Nombre
FROM EMPLEADOS e 
JOIN REPARTO r
	ON e.CodE = r.CodE
JOIN BARES b 
	ON r.CodB = b.CodB
WHERE b.Nombre = 'Stop' and r.Fecha 
	BETWEEN '10/17/2005' and '10/23/05';

In [ ]:
#2. Obtener el Cif y nombre de los bares a los que se ha repartido cerveza de tipo **Botella** y capacidad inferior a 1 litro, ordenados por localidad.

SELECT DISTINCT
	b.Cif,
	b.Nombre
FROM BARES b 
JOIN REPARTO r
	ON b.CodB = r.CodB
JOIN CERVEZAS c 
	ON c.CodC = r.CodC
WHERE c.Envase  = 'Botella' and c.Capacidad < 1
ORDER BY b.Localidad;

In [ ]:
# 3.Obtener los repartos (nombre del bar, envase y capacidad de la bebida, fecha y cantidad) realizados por **Prudencio Caminero**.

SELECT DISTINCT  
	b.Nombre,
	c.Envase,
	c.Capacidad,
	r.Fecha,
	r.Cantidad
FROM BARES b
JOIN REPARTO r
	ON b.CodB = r.CodB
JOIN CERVEZAS c 
	ON c.CodC = r.CodC
JOIN EMPLEADOS e 
	ON e.CodE = r.CodE 	
WHERE e.Nombre = 'Prudencio Caminero';

In [ ]:
#4.Obtener los bares a los que se les ha repartido envases de tipo **botella** y capacidad 0.2 ó 0.33.

SELECT DISTINCT  
	b.Nombre
FROM BARES b
JOIN REPARTO r 
	ON b.CodB = r.CodB
JOIN CERVEZAS c 
	ON c.CodC = r.CodC
WHERE c.Envase = 'Botella' and c.Capacidad >= 0.2 and c.Capacidad <= 0.33;

In [ ]:
# 5. Nombre de los empleados que han repartido a los bares **"Stop"** y **"Las Vegas"** cervezas con envase botella.

SELECT e.Nombre
FROM EMPLEADOS e
JOIN REPARTO r
    ON e.CodE = r.CodE
JOIN CERVEZAS c
    ON c.CodC = r.CodC
JOIN BARES b
    ON r.CodB = b.CodB
WHERE c.Envase = 'Botella'
    AND b.Nombre IN ('Stop', 'Las Vegas')
GROUP BY e.CodE, e.Nombre
HAVING COUNT(DISTINCT b.CodB) = 2;
	
	

In [ ]:
# 6. Obtener el nombre y número de viajes que ha realizado cada empleado fuera de **Villa Botijo**.

SELECT 
	e.Nombre,
	COUNT(*)
FROM EMPLEADOS e 
JOIN REPARTO r
	ON e.CodE = r.CodE
JOIN BARES b 
	ON b.CodB = r.CodB
WHERE b.Localidad != 'Villa Botijo'
GROUP BY e.Nombre;

In [ ]:
#7.Obtener el nombre y localidad del bar que más litros de cerveza ha comprado.
SELECT
  	b.Nombre,
    b.Localidad,
    SUM(c.Capacidad * r.Cantidad) AS Cantidad_Litros
FROM BARES b
JOIN REPARTO r ON b.CodB = r.CodB
JOIN CERVEZAS c ON r.CodC = c.CodC
GROUP BY b.Nombre, b.Localidad
ORDER BY Cantidad_Litros DESC
LIMIT 1;

In [ ]:
# 8. Obtener los bares que han adquirido todos los tipos de cerveza con envase de botella y capacidad menor que 1 litro.

SELECT
	b.Nombre,
	COUNT(DISTINCT c.CodC)  	
FROM BARES b 

JOIN REPARTO r ON r.CodB = b.CodB 
JOIN CERVEZAS c ON r.CodC = c.CodC

WHERE c.Envase = 'Botella' and c.Capacidad < 1
GROUP BY b.Nombre

HAVING COUNT(DISTINCT c.CodC) = (
    SELECT COUNT(*)
    FROM CERVEZAS
    WHERE Envase = 'Botella'
      AND Capacidad < 1);

In [ ]:
# 9. * Subir un 5% el sueldo del empleado que más días haya trabajado.
UPDATE EMPLEADOS
SET Sueldo = Sueldo * 1.05
WHERE CodE IN (
    SELECT e.CodE
    FROM EMPLEADOS e
    JOIN REPARTO r ON r.CodE = e.CodE
    GROUP BY e.CodE
    HAVING COUNT(DISTINCT r.Fecha) = (
        SELECT MAX(Dias_Trabajados)
        FROM (
            SELECT
                CodE,
                COUNT(DISTINCT Fecha) AS Dias_Trabajados
            FROM REPARTO
            GROUP BY CodE
        ) AS Resultado));

In [ ]:
#10. Insertar un nuevo reparto del empleado **"Vicente Merario"** al bar **"Stop"** de 48 cervezas de tipo lata el día 10/26/05.
SELECT
	e.CodE
FROM EMPLEADOS e 
WHERE e.Nombre = 'Vicente Merario';

SELECT 
	b.CodB
FROM BARES b 
WHERE b.Nombre = 'Stop';

SELECT 
	c.CodC
FROM CERVEZAS c 
WHERE c.Envase = 'Lata';

INSERT INTO REPARTO (CodE, CodB, CodC, Fecha, Cantidad)
VALUES
(2, '001', '03', '10/26/05', 48);

SELECT *
FROM REPARTO;